In [1]:
import numpy as np
import pandas as pd
import csv

In [2]:
string="Class,Alcohol,Malic acid,Ash,Alcalinity of ash  ,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline"        
cols=string.split(',')

In [3]:
df=pd.read_csv('data/wine.csv',sep=',',header=None,names=cols,quoting=csv.QUOTE_NONE)

In [4]:
df['Class']=df['Class'].str.replace('"','').astype('int64')
df['Proline']=df['Proline'].str.replace('"','').astype('int64')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Class                         178 non-null    int64  
 1   Alcohol                       178 non-null    float64
 2   Malic acid                    178 non-null    float64
 3   Ash                           178 non-null    float64
 4   Alcalinity of ash             178 non-null    float64
 5   Magnesium                     178 non-null    int64  
 6   Total phenols                 178 non-null    float64
 7   Flavanoids                    178 non-null    float64
 8   Nonflavanoid phenols          178 non-null    float64
 9   Proanthocyanins               178 non-null    float64
 10  Color intensity               178 non-null    float64
 11  Hue                           178 non-null    float64
 12  OD280/OD315 of diluted wines  178 non-null    float64
 13  Proline         

In [6]:
df.isna().sum(axis=0)

Class                           0
Alcohol                         0
Malic acid                      0
Ash                             0
Alcalinity of ash               0
Magnesium                       0
Total phenols                   0
Flavanoids                      0
Nonflavanoid phenols            0
Proanthocyanins                 0
Color intensity                 0
Hue                             0
OD280/OD315 of diluted wines    0
Proline                         0
dtype: int64

In [7]:
df.head()

,Class,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [8]:
df.columns

Index(['Class', 'Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash  ',
       'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols',
       'Proanthocyanins', 'Color intensity', 'Hue',
       'OD280/OD315 of diluted wines', 'Proline'],
      dtype='str')

In [9]:
df.shape

(178, 14)

In [10]:
X,y=df.iloc[:,1:],df.iloc[:,0]

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [12]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [13]:
from models import SoftMax

Before PCA

In [14]:
SM=SoftMax(eps=1e-5,max_iter=1000,lr=1,l2=1e-2)
SM.fit(X_train,y_train)
preds=SM.predict(X_test)
print(f"Accuracy: {np.mean(preds==y_test)*100}%")

total iterations: 515
Accuracy: 100.0%


In [15]:
class PCA:
    def fit(self,X,k=3,p=None):
        cov=np.cov(X,rowvar=False)
        e_vals,e_vecs=np.linalg.eigh(cov)# eigh since cov.T=cov
        self.e_vals,self.e_vecs=e_vals[::-1],e_vecs[:,::-1]
        self.evc=self.e_vals/np.sum(self.e_vals)
        self.evc_cum_sum=np.cumsum(self.evc)
        if p!=None:
            self.min_k=(np.argwhere((self.evc_cum_sum)>=p).ravel()[0])+1
            k=self.min_k
        self.W=self.e_vecs[:, :k]
               
    def transform(self,X):
        return X.dot(self.W)

In [16]:
pca=PCA()
pca.fit(X_train,p=0.6)
X_train_pca=pca.transform(X_train)
X_test_pca=pca.transform(X_test)

In [17]:
print(X_train.shape,X_train_pca.shape)

(142, 13) (142, 3)


After PCA

In [18]:
SM=SoftMax(eps=1e-5,max_iter=1000,lr=1,l2=1e-2)
SM.fit(X_train_pca,y_train)
preds=SM.predict(X_test_pca)
print(f"Accuracy: {np.mean(preds==y_test)*100}%")

total iterations: 227
Accuracy: 100.0%


Same accuracy with only 3 features.